# SME Credit Scoring — итоговый отчёт

**Logistic Regression · CatBoost · LightGBM | Raw · Platt · Isotonic**

**Цель:** проверить воспроизводимый ML-конвейер на синтетических данных МСП. Сравниваем ранжирование риска (ROC-AUC, Gini, PR-AUC) и качество вероятностей (Brier, Log Loss, calibration curves).

> **Ограничение:** в демонстрационном режиме метки `default_12m` **симулируются**, а не наблюдаются после выдачи кредитов. Поэтому численные результаты **не подтверждают** реальную способность банка предсказывать дефолты. Этот ноутбук не подбирает гиперпараметры и не обучает модели на тестовой части.

## 1. Конфигурация и загрузка

Ноутбук **не обучает модели повторно**. Он читает артефакты, которые записал `ml/scoring/main.py` (или прежний `calibrate_scoring.py`).

При необходимости укажи точную директорию в `RUN_DIR` или перед запуском установи переменную окружения `SCORING_RUN_DIR`.

In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, brier_score_loss, log_loss,
)
from IPython.display import display

DEFAULT_RUNS = [
    Path('ml/scoring/data/experiment_43/scoring'),
    Path('ml/scoring/data/calibration_43'),
]
if os.environ.get('SCORING_RUN_DIR'):
    RUN_DIR = Path(os.environ['SCORING_RUN_DIR']).expanduser()
else:
    RUN_DIR = next((p for p in DEFAULT_RUNS if (p / 'calibration_comparison.csv').exists()), DEFAULT_RUNS[0])
RUN_DIR = RUN_DIR.resolve()
FILES = ['calibration_comparison.csv', 'test_predictions_calibrated.csv',
         'split_manifest.csv', 'calibration_comparison.json']
missing = [name for name in FILES if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing {missing} in {RUN_DIR}. Run ml/scoring/main.py first or set SCORING_RUN_DIR.')

summary = pd.read_csv(RUN_DIR / 'calibration_comparison.csv')
preds = pd.read_csv(RUN_DIR / 'test_predictions_calibrated.csv', dtype={'application_id': str, 'borrower_id': str})
splits = pd.read_csv(RUN_DIR / 'split_manifest.csv', dtype={'application_id': str, 'borrower_id': str})
metadata = json.loads((RUN_DIR / 'calibration_comparison.json').read_text(encoding='utf-8'))
FIG_DIR = RUN_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_DIR)
print('Label source:', metadata.get('label_source', 'not specified'))
print('Models:', ', '.join(summary.model.unique()))

## 2. Sanity checks: разбиение, целевая переменная, согласованность метрик

Проверяем, что нет общих заёмщиков между train, validation, calibration и test; каждая строка предсказаний соответствует test. Эти проверки **не заменяют аудита временной утечки на уровне исходных источников**, но предотвращают ряд распространённых ошибок.

In [ ]:
assert not preds['application_id'].duplicated().any(), 'Duplicate predictions'
assert not splits['application_id'].duplicated().any(), 'Duplicate split applications'
assert not splits['borrower_id'].isna().any(), 'Missing borrower IDs'
assert set(preds['default_12m'].unique()).issubset({0, 1}), 'Invalid target'
assert set(splits.loc[splits['split']=='test', 'application_id']) == set(preds['application_id']), 'Test-ID mismatch'
assert set(splits['split'].unique()) == {'train', 'val', 'calibration', 'test'}, 'Unexpected split naming'
borrower_split_counts = splits.groupby('borrower_id')['split'].nunique()
assert borrower_split_counts.max() == 1, 'Borrower leakage across train/val/calibration/test'
assert (preds[['borrower_id','application_id']].merge(
    splits[['borrower_id','application_id','split']], on=['borrower_id','application_id'], validate='one_to_one'
)['split'] == 'test').all()
models = [m for m in ('logistic','catboost','lightgbm') if m in summary.model.unique()]
methods = [m for m in ('raw','platt','isotonic') if m in summary.calibration.unique()]
for model in models:
    for method in methods:
        if (summary.model.eq(model) & summary.calibration.eq(method)).any():
            col = f'{model}_{method}'
            assert col in preds, f'Missing probabilities: {col}'
            assert preds[col].notna().all() and preds[col].between(0,1).all(), f'Invalid probabilities: {col}'
assert preds['default_12m'].nunique() == 2, 'Need both classes'

print('Splits:', splits['split'].value_counts().to_dict())
print('Borrowers in >1 split:', int((borrower_split_counts > 1).sum()))
print('Test:', len(preds), 'rows; simulated defaults:', int(preds.default_12m.sum()),
      f'({preds.default_12m.mean():.1%})')
print('Predicted probability columns:', len([c for c in preds if c.endswith(('_raw','_platt','_isotonic'))]))

## 3. Сравнение метрик на test

- **ROC-AUC:** вероятность правильно ранжировать случайную пару «дефолт / не дефолт».
- **Gini = 2 × ROC-AUC − 1.**
- **Average Precision (AP):** summary по Precision–Recall; baseline примерно равен доле дефолтов.
- **Brier:** среднеквадратичная ошибка вероятностей, меньше лучше.
- **Log loss:** штрафует особенно уверенные ошибочные прогнозы, меньше лучше.

Числа ниже относятся **только к выбранной симуляции**, а не к реальным заёмщикам.

In [ ]:
columns = ['model','calibration','roc_auc','gini','average_precision','brier','log_loss','event_rate','mean_prediction']
metrics = summary[columns].sort_values(['model','calibration']).reset_index(drop=True)
display(metrics.round(4))
print('Brier: raw -> Platt -> Isotonic')
display(summary.pivot(index='model',columns='calibration',values='brier').reindex(columns=methods).round(5))

## 4. ROC-кривые: ранжирование риска

Отображаем **исходные** прогнозы (`raw`). Platt — строго монотонная калибровка, поэтому обычно сохраняет ROC-AUC; isotonic может создавать одинаковые прогнозы и менять его.

In [ ]:
y = preds['default_12m'].to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(8.5,5.5))
for name in models:
    p = preds[f'{name}_raw'].to_numpy(dtype=float)
    fpr,tpr,_ = roc_curve(y,p)
    ax.plot(fpr,tpr,lw=2,label=f'{name}: AUC={roc_auc_score(y,p):.3f}, Gini={2*roc_auc_score(y,p)-1:.3f}')
ax.plot([0,1],[0,1],'--',c='gray',lw=1,label='Random ranking')
ax.set(xlabel='False Positive Rate',ylabel='True Positive Rate',title='Test ROC — synthetic defaults')
ax.grid(alpha=.25); ax.legend(loc='lower right');fig.tight_layout()
fig.savefig(FIG_DIR/'roc_test.png',dpi=150);plt.show()

## 5. Precision–Recall: почему кривая ступенчатая?

Используем `step`, а не обычную интерполяцию линией. Переход к следующей положительной заявке изменяет recall на `1 / число_дефолтов`, поэтому при небольшом числе событий неизбежны заметные скачки. Начальная/конечная технические точки PR-кривой не гарантируют реального precision 100%.

In [ ]:
fig,ax = plt.subplots(figsize=(8.5,5.5))
for name in models:
    p = preds[f'{name}_raw'].to_numpy(dtype=float)
    precision, recall, _ = precision_recall_curve(y,p)
    ax.step(recall[::-1],precision[::-1],where='post',lw=1.8,
            label=f'{name}: AP={average_precision_score(y,p):.3f}')
ax.axhline(y.mean(),ls='--',c='gray',label=f'Event prevalence: {y.mean():.3f}')
ax.set(xlabel='Recall',ylabel='Precision',title='Test Precision–Recall — raw scores',xlim=(0,1),ylim=(0,1.03))
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(FIG_DIR/'precision_recall_test.png',dpi=150);plt.show()
print('Recall jump for one more identified event:',f'{1/y.sum():.2%}')

## 6. Калибровка вероятностей с доверительными интервалами

Калибровочная диагональ означает совпадение прогнозируемого риска и частоты события. Каждая точка агрегирует группу прогнозов; при небольшом количестве дефолтов доверительные интервалы могут быть широкими.

Здесь используем корзины по квантилям, не разделяя одинаковые значения вероятностей. **Границы Wilson 95%** отражают биномиальную неопределённость внутри корзины; они не учитывают несопоставимость искусственного таргета с реальными дефолтами.

In [ ]:
def wilson_interval(events, total, z=1.95996398454):
    phat = events/total
    denom = 1+z*z/total
    center = (phat+z*z/(2*total))/denom
    half = z*np.sqrt(phat*(1-phat)/total+z*z/(4*total**2))/denom
    return np.maximum(0,center-half),np.minimum(1,center+half)

def reliability_bins(target, probabilities, n_bins=7):
    frame = pd.DataFrame({'y':target,'p':probabilities})
    if frame.p.nunique() == 1:
        frame['bin'] = 0
    else:
        frame['bin'] = pd.qcut(frame.p,q=min(n_bins, len(frame)),duplicates='drop',labels=False)
    result = frame.groupby('bin',observed=True).agg(n=('y','size'),events=('y','sum'),
        predicted=('p','mean'), observed=('y','mean')).reset_index()
    return result

fig, axes = plt.subplots(1,len(models),figsize=(5.1*len(models),4.8),sharex=True,sharey=True,squeeze=False)
for ax,name in zip(axes.ravel(),models):
    for method in methods:
        if f'{name}_{method}' not in preds: continue
        group = reliability_bins(y,preds[f'{name}_{method}'].to_numpy(dtype=float))
        lo,hi = wilson_interval(group.events.to_numpy(),group.n.to_numpy())
        ax.errorbar(group.predicted,group.observed,
                    yerr=[group.observed.to_numpy()-lo,hi-group.observed.to_numpy()],
                    marker='o',capsize=2,label=method,linewidth=1)
    ax.plot([0,1],[0,1],'--',color='gray',linewidth=1)
    ax.set_title(name)
    ax.set(xlabel='Mean predicted P(default)',xlim=(0,.7),ylim=(0,.7))
    ax.grid(alpha=.2);ax.legend()
axes.ravel()[0].set_ylabel('Observed synthetic default rate')
fig.suptitle('Reliability on test; Wilson 95% intervals')
fig.tight_layout();fig.savefig(FIG_DIR/'calibration_intervals_test.png',dpi=150);plt.show()

## 7. Проверяем сжатие диапазона прогнозов

Модели, остановившиеся на нескольких деревьях, могут выдавать почти одинаковые вероятности многим объектам. Это ограничивает полезность индивидуальных оценок риска, даже если ROC-AUC выглядит неплохо.

In [ ]:
fig, axes = plt.subplots(1,len(models),figsize=(4.9*len(models),4.1),sharey=True,squeeze=False)
for ax,name in zip(axes.ravel(),models):
    for method in ('raw','platt'):
        if f'{name}_{method}' not in preds: continue
        probabilities = preds[f'{name}_{method}'].to_numpy(dtype=float)
        ax.hist(probabilities,bins=20,alpha=.5,label=f'{method}, unique={len(np.unique(probabilities))}')
    ax.axvline(y.mean(),ls='--',c='gray',label='Observed event rate')
    ax.set(title=name,xlabel='Predicted probability')
    ax.grid(alpha=.2);ax.legend(fontsize=8)
axes.ravel()[0].set_ylabel('Number of applications')
fig.tight_layout();fig.savefig(FIG_DIR/'score_distributions.png',dpi=150);plt.show()

## 8. Эффект Platt / isotonic и статистическая неопределённость

Считаем `Brier(raw) − Brier(calibrated)`: положительное значение — улучшение. Если доверительный интервал пересекает ноль, по этой выборке улучшение нельзя считать устойчивым. Напомним: методы были заранее заданы; выбирать метод по test и снова сообщать метрику на том же test — смещённая оценка.

In [ ]:
cols = ['model','calibration','brier','brier_gain_vs_raw','brier_gain_ci_low','brier_gain_ci_high']
if set(cols).issubset(summary.columns):
    gains = summary.loc[summary.calibration!='raw',cols].copy()
    display(gains.round(5))
else:
    print('Paired bootstrap intervals not found in this experiment. Report only point changes.')

## 9. Итоги и ограничения

**Что можно говорить:**

- сравнивали Logistic Regression, CatBoost и LightGBM на одном test;
- калибраторы обучались на отдельной calibration-части, не на test;
- протокол включает защиту от пересечения заёмщиков и исключает признаки после решения банка;
- сложный бустинг не обязательно лучше простой модели по Brier / PR-AUC.

**Чего говорить нельзя:** «банк будет иметь ROC-AUC X», «высокий риск означает доказанный дефолт», «калибратор выдаёт реальные PD».

**Следующий этап:** независимые повторные синтетические эксперименты, затем — обучение/валидация на реальных наблюдаемых исходах с временным разбиением и аудитом доступности каждого признака на дату заявки.